In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Tue Apr 14 22:51:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/cmpe281_data"

# Model Pre-Processing

In [ ]:
import os
from collections import defaultdict

def parse_labels(split):
    label_dir = os.path.join(DATASET_PATH, split, "labels")

    class_counts = defaultdict(int)
    total_labels = 0

    if not os.path.exists(label_dir):
        print(f"Missing folder: {label_dir}")
        return

    for file in os.listdir(label_dir):
        if not file.endswith(".txt"):
            continue

        file_path = os.path.join(label_dir, file)

        with open(file_path, "r") as f:
            lines = f.readlines()

            for line in lines:
                parts = line.strip().split()

                # validate YOLO format
                if len(parts) != 5:
                    continue

                try:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1
                    total_labels += 1
                except:
                    continue

    print(f"\n=== {split.upper()} SET ===")
    print("Total labels:", total_labels)
    print("Class distribution:")
    for k, v in sorted(class_counts.items()):
        print(f"Class {k}: {v}")


# 🚀 run for all splits
for split in ["train", "valid", "test"]:
    parse_labels(split)


=== TRAIN SET ===
Total labels: 22028
Class distribution:
Class 0: 22028

=== VALID SET ===
Total labels: 2132
Class distribution:
Class 0: 2132

=== TEST SET ===
Total labels: 1048
Class distribution:
Class 0: 1048


In [ ]:
from PIL import Image

img_path = "/content/drive/MyDrive/cmpe281_data/train/images/00d17e785bbf2ca1_jpg.rf.505336abcb2700ae8354dc15bf58794d.jpg"

img = Image.open(img_path)
print("Size (width, height):", img.size)

Size (width, height): (640, 640)


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00


In [ ]:
data_yaml = """
path: /content/drive/MyDrive/cmpe281_data
train: train/images
val: valid/images
test: test/images

nc: 3
names: ["class0", "class1", "class2"]
"""

with open("/content/drive/MyDrive/cmpe281_data/data.yaml", "w") as f:
    f.write(data_yaml)

# Model Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # small + fast model

model.train(
    data="/content/drive/MyDrive/cmpe281_data/data.yaml",

    # training
    epochs=100,
    imgsz=512,
    batch=16,
    cache="disk",         # cache images

    # speed tuning
    workers=2,
    amp=True,

    # output
    project="/content/drive/MyDrive/yolo_runs",
    name="exp1",
    exist_ok=True,

    # checkpointing
    save=True,
    save_period=5      # save every 5 epoch
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cmpe281_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c08467b4170>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_runs/exp1/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


#Model Testing

In [ ]:
model.predict(
    source="/content/drive/MyDrive/cmpe281_data/test/images",
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1019 /content/drive/MyDrive/cmpe281_data/test/images/0002a5b67e5f0909_jpg.rf.07ca41e79eb878b14032f650f34d0967.jpg: 512x512 2 class0s, 7.7ms
image 2/1019 /content/drive/MyDrive/cmpe281_data/test/images/000812dcf304a8e7_jpg.rf.559f904bc045f68ee947796a1b561d8f.jpg: 512x512 1 class0, 9.9ms
image 3/1019 /content/drive/MyDrive/cmpe281_data/test/images/0010f4c10f7ab07e_jpg.rf.92344aa620e23aacc490273e32343595.jpg: 512x512 1 class0, 9.7ms
image 4/1019 /co

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'class0', 1: 'class1', 2: 'class2'}
 obb: None
 orig_img: array([[[ 49,  75,  62],
         [ 42,  68,  55],
         [ 34,  58,  46],
         ...,
         [223, 213, 195],
         [223, 213, 195],
         [223, 213, 195]],
 
        [[ 49,  73,  61],
         [ 45,  69,  57],
         [ 40,  64,  52],
         ...,
         [223, 213, 195],
         [223, 213, 195],
         [223, 213, 195]],
 
        [[ 50,  72,  60],
         [ 53,  75,  63],
         [ 53,  76,  62],
         ...,
         [223, 213, 195],
         [223, 213, 195],
         [223, 213, 195]],
 
        ...,
 
        [[ 75,  83,  83],
         [ 75,  83,  83],
         [ 77,  82,  83],
         ...,
         [175, 171, 166],
         [174, 170, 165],
         [174, 170, 165]],
 
        [[ 84,  89,  90],
         [ 84,  89,  90],
         [ 83,  89,  88],
      

# Model Evaluation

In [ ]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.1±0.0 MB/s, size: 55.0 KB)
val: Scanning /content/drive/MyDrive/cmpe281_data/valid/labels.cache... 2046 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2046/2046 343.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 1.4it/s 1:32
                   all       2046       2132       0.98      0.962      0.981      0.696
                class0       2043       2132       0.98      0.962      0.981      0.696
Speed: 0.5ms preprocess, 0.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultral

# Text Extraction

## Config

In [ ]:
!pip install easyocr

In [124]:
from ultralytics import YOLO
import easyocr
import cv2
import os
import numpy as np

# ====== CONFIG ======
MODEL_PATH = "/content/drive/MyDrive/yolo_runs/exp1/weights/best.pt"
IMAGE_FOLDER = "/content/drive/MyDrive/cmpe281_data/sample"
OUTPUT_FOLDER = "/content/drive/MyDrive/cmpe281_data/ocr_results"
DEBUG_FOLDER = "/content/drive/MyDrive/cmpe281_data/debug_boxes"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(DEBUG_FOLDER, exist_ok=True)

In [125]:
model = YOLO(MODEL_PATH)
reader = easyocr.Reader(['en'])

## Function Defintions

In [ ]:
import os
import cv2
import json
import numpy as np


import cv2
import numpy as np

def preprocess_plate(plate_crop):
    if plate_crop is None or plate_crop.size == 0:
        return None

    # 1. GENTLE CROP
    # The current crop is too tight, cutting into the L.
    # Let's reduce the margin to 5% to keep the full characters.
    h, w = plate_crop.shape[:2]
    margin_v, margin_h = int(h * 0.05), int(w * 0.05)
    plate_crop = plate_crop[margin_v:h-margin_v, margin_h:w-margin_h]

    # 2. SHARP RESIZE
    target_h = 150 # Keeping it smaller to prevent "L-block" syndrome
    ratio = target_h / float(plate_crop.shape[0])
    resized = cv2.resize(plate_crop, (int(plate_crop.shape[1] * ratio), target_h),
                         interpolation=cv2.INTER_LANCZOS4)

    # 3. GRAYSCALE + BILATERAL (The shadow killer)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    # Bilateral filter smooths the plate surface but keeps the letter edges sharp
    denoised = cv2.bilateralFilter(gray, 11, 17, 17)

    return denoised

# Run

In [126]:
# SORT TEST CASES
import re

def extract_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else -1

image_files = sorted(
    os.listdir(IMAGE_FOLDER),
    key=extract_number
)

In [127]:

# =========================
# MAIN PIPELINE
# =========================
for img_name in image_files:
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg", ".heic")):
        continue

    img_path = os.path.join(IMAGE_FOLDER, img_name)
    img = cv2.imread(img_path)
    name, _ = os.path.splitext(img_name)
    vis_img = img.copy()

    results = model.predict(img_path, conf=0.25, verbose=False)
    boxes = results[0].boxes.xyxy if results[0].boxes is not None else []

    print(f"\n📷 {name}")
    extracted_texts = []

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)

        # Initial crop from YOLO
        raw_crop = img[y1:y2, x1:x2]

        # 1. APPLY THE NEW PREPROCESSING
        proc = preprocess_plate(raw_crop)

        if proc is None:
            continue

        cv2.imwrite(f"{DEBUG_FOLDER}/{name}_box{i}_proc.jpg", proc)

        # 2. RUN OCR
        result = reader.readtext(
            proc,
            allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
            beamWidth=10,  # Increase from default 5 to 10 or 15
            batch_size=1,
            detail=1,
            workers=0
        )

        if result:
            # 3. FOCUS ON BIGGEST TEXT (Sort by Bounding Box Area)
            # EasyOCR returns: [([[x,y], [x,y], [x,y], [x,y]], text, confidence), ...]
            result.sort(key=lambda x: (x[0][1][0] - x[0][0][0]) * (x[0][2][1] - x[0][1][1]), reverse=True)

            # The largest area is likely the actual license number
            plate, conf = result[0][1], result[0][2]
            plate = plate.replace(" ", "").strip()

            if len(plate) >= 4 and conf > 0.5:
                # =========================
                # VISUALIZATION & STORAGE
                # =========================
                color = (0, 255, 0)
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), color, 2)
                label = f"{plate} ({conf:.2f})"

                cv2.putText(vis_img, label, (x1, max(0, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                extracted_texts.append({
                    "plate": plate,
                    "confidence": round(float(conf), 3)
                })
                print(f"  → {plate} (conf: {conf:.2f})")

    # SAVE OUTPUTS (JSON & Image)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    with open(os.path.join(OUTPUT_FOLDER, img_name + ".json"), "w") as f:
        json.dump({"status": "OK" if extracted_texts else "MISSING", "results": extracted_texts}, f, indent=2)

    cv2.imwrite(os.path.join(DEBUG_FOLDER, f"{img_name}_vis.jpg"), vis_img)


📷 test1
  → TIEESUK (conf: 0.59)

📷 test2
  → LAPDCGC (conf: 0.90)

📷 test3

📷 test4

📷 test5

📷 test6
  → ISP666 (conf: 0.93)

📷 test7

📷 test8

📷 test9
  → BAXJL1ZI (conf: 0.55)

📷 test10

📷 test11

📷 test12
  → 8AXJL1Z (conf: 0.73)
  → 6ASL837 (conf: 0.59)

📷 test13
  → BAXJL17 (conf: 0.68)

📷 test14

📷 test15
  → 8AXJLIZ (conf: 0.72)

📷 test16

📷 test17

📷 test18
  → BAXJL1Z (conf: 0.73)

📷 test19
  → BAXJL1Z (conf: 0.66)
